# Kiểm tra bản sao 292p của tập val: có cho kết quả như video gốc không?

**Add Input:**
1. Hai dataset 292p của tập val (`ek100-292-val-0`, `ek100-292-val-1`, tạo từ output của `kaggle_reencode_ek100`).
2. Output của lần chạy `kaggle_eval_ek100_original` **đã đủ 138 video** (có thư mục `logits_orig`), dùng để so sánh.

**Cài đặt:** GPU T4 x2. Internet On (để tải checkpoint). Mất khoảng 1–1.5 giờ: video nhỏ nên decode nhanh, phần chậm nhất là GPU.

**Kỳ vọng:** recall của bản 292p gần với video gốc (official 32.71, clean 31.23 ở giao thức gốc; clean 14.63 ở giao thức chuẩn).
Tập top-5 trùng phần lớn clip. Nếu khớp, ta dùng bản sao 292p cho toàn bộ phần train sau này mà không phải tải lại video gốc.

In [ ]:
BRANCH = "claude/vjepa2-verify-improve-dsfgtg"
VJEPA2_COMMIT = "204698b45b3712590f06245fbfba32d3be539812"  # version the code was checked against
!git clone -q https://github.com/1Tyson/Testing-And-Research-VJEPA.git -b $BRANCH /kaggle/working/repo
!git clone -q https://github.com/facebookresearch/vjepa2.git /kaggle/working/vjepa2 && cd /kaggle/working/vjepa2 && git checkout -q $VJEPA2_COMMIT
!pip install -q decord webdataset timm einops
import os
os.environ["VJEPA2_ROOT"] = "/kaggle/working/vjepa2"
R = "/kaggle/working/repo"
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# Checkpoints (~1.3GB encoder+predictor, probe) -> /tmp so they do not count toward the 20GB output limit
!mkdir -p /tmp/ckpt
!wget -q -nc -P /tmp/ckpt https://dl.fbaipublicfiles.com/vjepa2/vitl.pt
!wget -q -nc -P /tmp/ckpt https://dl.fbaipublicfiles.com/vjepa2/evals/ek100-vitl-256.pt
!ls -lh /tmp/ckpt

In [ ]:
# Official annotations (the full train csv is required: probe output classes are derived from it)
!git clone -q --depth 1 https://github.com/epic-kitchens/epic-kitchens-100-annotations.git /kaggle/working/ek100-ann
!ls /kaggle/input; find /kaggle/input -maxdepth 3 | head -30

In [ ]:
import glob, os, shutil, json
mp4 = glob.glob("/kaggle/input/**/ek100_292/**/*.MP4", recursive=True)
manifests = glob.glob("/kaggle/input/**/ek100_292/manifest.csv", recursive=True)
print(len(mp4), "re-encoded videos in", len(manifests), "dataset(s)")
assert len(mp4) >= 138, "attach both ek100-292-val datasets"
ref = glob.glob("/kaggle/input/**/logits_orig", recursive=True)
assert ref, "attach the output of the complete kaggle_eval_ek100_original run (folder logits_orig)"
REF = ref[0]; print("reference logits:", REF)
VROOT = os.path.commonpath([os.path.dirname(os.path.dirname(os.path.dirname(p))) for p in mp4])
print("video root:", VROOT)
# fps/length from the official video info (the copies are 292p, so a resolution check against it would fail)
!python $R/scripts/prepare_ek100.py --video_root $VROOT \
    --train_csv /kaggle/working/ek100-ann/EPIC_100_train.csv \
    --val_csv /kaggle/working/ek100-ann/EPIC_100_validation.csv \
    --video_info /kaggle/working/ek100-ann/EPIC_100_video_info.csv \
    --out_dir /kaggle/working/work | grep -E '"val_clips"|val_videos_found|out_of_range_clips"'

In [ ]:
# Eval on the copies: both anchors, 2 GPUs (resumable)
import subprocess, time
OUT = "/kaggle/working/logits_292"
def shard_cmd(gpu):
    run = (f"python {R}/scripts/eval_ek100.py --work_dir /kaggle/working/work --out_dir {OUT} "
           f"--vitl_ckpt /tmp/ckpt/vitl.pt --probe_ckpt /tmp/ckpt/ek100-vitl-256.pt --anchors official action_start "
           f"--device cuda:{gpu} --shard_id {gpu} --num_shards 2 --dtype fp16 --num_workers 2")
    return f"for i in 1 2 3 4 5; do {run} && break; echo \"[retry $i]\"; done"
procs = {g: subprocess.Popen(["bash", "-c", shard_cmd(g) + f" >> /kaggle/working/eval292_gpu{g}.log 2>&1"]) for g in (0, 1)}
while any(p.poll() is None for p in procs.values()):
    time.sleep(120)
    for g in procs:
        lines = [l for l in open(f"/kaggle/working/eval292_gpu{g}.log").read().splitlines()
                 if any(k in l for k in ("clip/s", "[error]", "[retry", "to go", "Traceback"))]
        print(f"gpu{g}: {lines[-1] if lines else 'loading model...'}", flush=True)
print("exit codes:", {g: p.returncode for g, p in procs.items()})

In [ ]:
# Same metric on the copies and on the originals, plus per-clip agreement of the logits
for anchor in ("official", "action_start"):
    print(f"\n=================== {anchor} ===================")
    print("--- 292p copy ---")
    !python $R/scripts/compute_metrics.py --work_dir /kaggle/working/work --logits_dir $OUT/{anchor} --out /kaggle/working/m292_{anchor}.json --allow_partial | tail -3
    print("--- original videos ---")
    !python $R/scripts/compute_metrics.py --work_dir /kaggle/working/work --logits_dir $REF/{anchor} --out /kaggle/working/morig_{anchor}.json --allow_partial | tail -3
    print("--- agreement (copy vs original) ---")
    !python $R/scripts/diagnose_ek100.py --work_dir /kaggle/working/work --logits_dir $OUT/{anchor} --compare_logits_dir $REF/{anchor} --bootstrap 5 --out /kaggle/working/cmp_{anchor}.json | grep -A6 '"compare"'

Gửi lại output của cell cuối.